In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("./iris/iris.csv")
statistical_analysis = df.describe()
k = 5
df

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


In [2]:
df.replace({"Species": {"Iris-setosa": 0, "Iris-versicolor": 1, "Iris-virginica": 2}}, inplace=True)
df.rename(
    columns={"PetalLengthCm": "petal_length", "PetalWidthCm": "petal_width", "SepalLengthCm": "sepal_length", "SepalWidthCm": "sepal_width", 'Id': 'id'},
    inplace=True,
)
df.sample(frac=1).reset_index(drop=True, inplace=True)

# training_df = df[df.index < df.shape[0] * 0.8].reset_index(drop=True)
# test_df = df[df.index >= df.shape[0] * 0.8].reset_index(drop=True)
training_df = df.sample(frac=0.8).reset_index(drop=True)
test_df = df.sample(frac=0.2).reset_index(drop=True)

In [3]:
for index, row in test_df.iterrows():
    distance_df = pd.DataFrame()
    distance_df['distance'] = np.sqrt(
        (training_df['sepal_length'] - row['sepal_length']) ** 2 +
        (training_df['sepal_width'] - row['sepal_width']) ** 2 +
        (training_df['petal_length'] - row['petal_length']) ** 2 +
        (training_df['petal_width'] - row['petal_width']) ** 2
    ).round(2)
    distance_df.sort_values(by='distance', inplace=True)
    # k is the selection factor, then we choose the most common species among the k nearest neighbors as the predicted species for the test data point.
    distance_df['species'] = training_df["Species"]
    result = distance_df.head(k)['species'].mode()[0]
    test_df.at[index, 'predicted_species'] = result

test_df['predicted_species'] = test_df['predicted_species'].astype(int)
print(test_df[['Species', 'predicted_species']])

   Species  predicted_species
0        0                  0
1        2                  2
2        0                  0
3        0                  0
4        1                  1
5        2                  2
6        0                  0
7        2                  2
8        1                  1
9        2                  2
10       0                  0
11       2                  2
12       1                  1
13       0                  0
14       2                  2
15       0                  0
16       2                  2
17       0                  0
18       2                  2
19       2                  2
20       1                  1
21       0                  0
22       2                  2
23       2                  1
24       0                  0
25       1                  2
26       1                  1
27       0                  0
28       1                  1
29       1                  1


In [ ]:
error = (test_df['Species'] != test_df['predicted_species']).sum() / test_df.shape[0]
print(f"Error: {error:.2f}")

Error: 0.07


id                   120
sepal_length         6.0
sepal_width          2.2
petal_length         5.0
petal_width          1.5
Species                2
predicted_species      1
Name: 23, dtype: object

In [15]:
# Confusion Matrix
true_positive = (
    test_df["Species"] == test_df["predicted_species"]
).sum()
false_negative = (
    test_df["Species"] != test_df["predicted_species"]
).sum()
print(f"True Positive: {true_positive}")
print(f"False Negative: {false_negative}")

True Positive: 28
False Negative: 2
